# Post Pipeline Verification (SQL)

This notebook verifies control, landing, silver, and audit outputs after the orchestrator finishes.
Run cells in order.

## Cell 1: Set Runtime Parameters

This cell sets catalog/schema parameters used by all verification queries.
You can override defaults via widgets when running manually or as a job task.

In [ ]:
dbutils.widgets.text('catalog', 'main')
dbutils.widgets.text('bronze_schema', 'bronze_dev')
dbutils.widgets.text('silver_schema', 'silver_dev')
dbutils.widgets.text('audit_schema', 'audit_dev')
dbutils.widgets.text('control_schema', 'control_dev')

catalog = dbutils.widgets.get('catalog').strip()
bronze_schema = dbutils.widgets.get('bronze_schema').strip()
silver_schema = dbutils.widgets.get('silver_schema').strip()
audit_schema = dbutils.widgets.get('audit_schema').strip()
control_schema = dbutils.widgets.get('control_schema').strip()

print('catalog      =', catalog)
print('bronze_schema=', bronze_schema)
print('silver_schema=', silver_schema)
print('audit_schema =', audit_schema)
print('control_schema=', control_schema)

## Cell 2: Helper To Run SQL

This cell defines a helper to execute SQL and display results consistently.

In [ ]:
def run_sql(title: str, query: str):
    print('\n===', title, '===')
    print(query)
    df = spark.sql(query)
    try:
        display(df)
    except NameError:
        df.show(truncate=False)
    return df

## Cell 3: Verify Control Tables Have Data

This checks row counts for source_registry, column_mapping, dq_rules, and publish_rules.
If these counts are zero, metadata bootstrap did not load correctly.

In [ ]:
q_control_counts = f"""
SELECT 'source_registry' AS table_name, COUNT(*) AS row_count
FROM {catalog}.{control_schema}.source_registry
UNION ALL
SELECT 'column_mapping', COUNT(*) FROM {catalog}.{control_schema}.column_mapping
UNION ALL
SELECT 'dq_rules', COUNT(*) FROM {catalog}.{control_schema}.dq_rules
UNION ALL
SELECT 'publish_rules', COUNT(*) FROM {catalog}.{control_schema}.publish_rules
"""
run_sql('Control table row counts', q_control_counts)

q_control_detail = f"DESCRIBE DETAIL {catalog}.{control_schema}.source_registry"
run_sql('Control source_registry storage detail', q_control_detail)

## Cell 4: Verify Files Picked From Connect Deltaload

This verifies how many distinct source files were picked and shows per-file row counts
from landing table metadata column _source_file.

In [ ]:
q_distinct_files = f"""
SELECT COUNT(DISTINCT _source_file) AS distinct_files_picked
FROM {catalog}.{bronze_schema}.connect_countryriskdet_landing
WHERE _source_file LIKE '%/deltaload/%'
"""
run_sql('Distinct connect deltaload files picked', q_distinct_files)

q_file_breakdown = f"""
SELECT _source_file, COUNT(*) AS rows_per_file
FROM {catalog}.{bronze_schema}.connect_countryriskdet_landing
WHERE _source_file LIKE '%/deltaload/%'
GROUP BY _source_file
ORDER BY rows_per_file DESC
"""
run_sql('Connect deltaload rows per file', q_file_breakdown)

## Cell 5: Verify Silver And Audit Outputs

This checks final silver row count and latest pipeline/audit events.

In [ ]:
q_silver_count = f"SELECT COUNT(*) AS silver_row_count FROM {catalog}.{silver_schema}.connect_countryriskdet"
run_sql('Silver row count (connect_countryriskdet)', q_silver_count)

q_pipeline_runs = f"""
SELECT run_id, source_system, source_entity, status, rows_in, rows_out, run_ts
FROM {catalog}.{audit_schema}.pipeline_runs
ORDER BY run_ts DESC
LIMIT 50
"""
run_sql('Latest pipeline runs', q_pipeline_runs)

q_dq_results = f"""
SELECT run_id, source_system, source_entity, dq_status, dq_failed_rule, row_count, run_ts
FROM {catalog}.{audit_schema}.dq_rule_results
ORDER BY run_ts DESC
LIMIT 100
"""
run_sql('Latest DQ results', q_dq_results)

q_lifecycle = f"""
SELECT entity_id, event_type, event_ts
FROM {catalog}.{audit_schema}.entity_lifecycle_log
ORDER BY event_ts DESC
LIMIT 50
"""
run_sql('Latest lifecycle events', q_lifecycle)